# 04 · Validate — symmetry-order vs success, tied vs untied, wrong-oligomer risk

**Standard slot:** *validate (in silico).* **For Project 04 this is the core analysis:** does
success depend on symmetry order? Does tying the sequence help? And — the scientific heart —
do top designs survive a **wrong-oligomer** check, or do they also score well as the wrong
order? (D3.)

Needs `results/predictions.csv` from notebook 03.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Symmetry order vs success

Compute the pass rate per symmetry under the `oligomer` filter (subunit scRMSD ≤ 2.5, pLDDT ≥ 80,
interface pAE ≤ 10) plus the symmetry-RMSD gate. Higher-order / dihedral assemblies are usually
harder — report the rates with N. (Mock numbers are SYNTHETIC `EXAMPLE_DATA`.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pred = pd.read_csv("results/predictions.csv")
C = dict(scrmsd=2.5, plddt=80, pae=10, sym=2.0)   # oligomer cutoffs + symmetry-RMSD gate

def passes(r):
    return (r["subunit_scrmsd"] <= C["scrmsd"] and r["plddt"] >= C["plddt"]
            and r["interface_pae"] <= C["pae"] and r["symmetry_rmsd"] <= C["sym"])
pred["pass"] = pred.apply(passes, axis=1)

by_sym = pred.groupby("symmetry").agg(n=("pass", "size"), n_pass=("pass", "sum"))
by_sym["success_rate"] = (by_sym["n_pass"] / by_sym["n"]).round(3)
print("symmetry order vs success (EXAMPLE_DATA on mock):")
print(by_sym)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(by_sym.index.astype(str), by_sym["success_rate"])
for i, (rate, n) in enumerate(zip(by_sym["success_rate"], by_sym["n"])):
    ax.text(i, rate, f"{rate}\n(N={n})", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("assembly-success rate"); ax.set_ylim(0, 1)
ax.set_title("Symmetry order vs success (EXAMPLE_DATA)")
plt.tight_layout(); plt.savefig("results/symmetry_vs_success.png", dpi=150); plt.show()

## 2 · Tied vs untied sequence design `[extension]`

Does tying symmetry-related positions improve the pass rate? Compare the tied and untied subsets
(the latter built in notebook 02). The expectation is that tying helps — untied chains can
diverge and break symmetry — but **report what your data actually show**, with N.

In [ ]:
if "tied" in pred.columns and pred["tied"].nunique() > 1:
    by_tied = pred.groupby("tied").agg(n=("pass", "size"), n_pass=("pass", "sum"))
    by_tied["success_rate"] = (by_tied["n_pass"] / by_tied["n"]).round(3)
    print("tied vs untied (EXAMPLE_DATA on mock):")
    print(by_tied)
else:
    print("Need both tied and untied designs in the pool (see notebook 02 step 4).")

## 3 · Wrong-oligomer risk analysis `[extension]`

The central failure mode. Two complementary views:
1. **Predicted-order mismatch** — `predicted_order` is the oligomeric order the predictor most
   prefers; if it differs from the intended `n_subunits`, the design is at risk.
2. **Alternative-state modelling** — predict the *same sequence* as other orders and check
   whether it scores comparably; a design that is 'happy' as the wrong order is a red flag.

On Colab, run `multimer_predict` with sequences arranged as different chain counts; here we use
the mock `predicted_order` and a small alternative-state sweep.

In [ ]:
# View 1: predicted-order mismatch.
pred["wrong_oligomer_risk"] = pred["predicted_order"] != pred["n_subunits"]
risk = pred.groupby("symmetry")["wrong_oligomer_risk"].mean().round(3)
print("fraction of designs whose PREFERRED order != intended order (EXAMPLE_DATA):")
print(risk)
print("\n-> high values mean the symmetry is hard to hit cleanly; treat those picks with caution.")

In [ ]:
# View 2: alternative-state sweep for the top design (model the sequence as several orders).
from sym_tools import multimer_predict, SYMMETRY_ORDER

pool = pd.read_csv("results/assemblies.csv")
top_row = pool.iloc[0]
seq, intended = top_row["sequence"], top_row["symmetry"]
ALT_STATES = ["C2", "C3", "C4", "D2"]   # candidate alternative orders to test
print(f"top design: intended {intended} ({SYMMETRY_ORDER[intended]} subunits)\n")
print(f"{'state':6s} {'order':6s} {'iface_pae':10s} {'sym_rmsd':9s}")
for st in ALT_STATES:
    s = multimer_predict(seq, st, tool="mock")   # -> tool="af2" on an A100
    flag = "  <-- intended" if st == intended else ""
    print(f"{st:6s} {SYMMETRY_ORDER[st]:<6d} {s.interface_pae:<10.2f} {s.symmetry_rmsd:<9.2f}{flag}")
print("\nRed flag: an ALTERNATIVE state scores as well as (or better than) the intended one.")

## 4 · Cross-metric view (interface pAE vs symmetry RMSD)

Interface pAE and symmetry RMSD measure different things. Plotting them together separates the
four quadrants: clean (both good), confident-but-wrong-arrangement (good pAE, bad symmetry RMSD),
and so on. The off-diagonal cases are exactly where wrong-oligomer risk hides.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for sym, g in pred.groupby("symmetry"):
    ax.scatter(g["interface_pae"], g["symmetry_rmsd"], label=str(sym), alpha=0.7)
ax.axvline(C["pae"], ls="--", lw=0.8, color="k")
ax.axhline(C["sym"], ls="--", lw=0.8, color="k")
ax.set_xlabel("interface pAE (Å)  — lower better"); ax.set_ylabel("symmetry RMSD (Å)  — lower better")
ax.set_title("Interface confidence vs symmetry closure (EXAMPLE_DATA)"); ax.legend(title="symmetry")
plt.tight_layout(); plt.savefig("results/pae_vs_symrmsd.png", dpi=150); plt.show()
print("Bottom-left quadrant = clean. Bottom-right of the vertical line = confident interface, wrong closure.")

## D3 (part 2) checklist
- [ ] Symmetry-order-vs-success rates with N (figure saved).
- [ ] Tied-vs-untied comparison reported honestly.
- [ ] Wrong-oligomer risk analysis: predicted-order mismatch **and** an alternative-state sweep on top picks.
- [ ] Interface-pAE-vs-symmetry-RMSD view; off-diagonal (confident-but-wrong) cases flagged.
- [ ] Failure-mode note: which symmetry is hardest, and why your top picks might still be wrong.

**Next:** `05_validation_plan.ipynb` — the nsEM/SEC-MALS plan + antigen-display extension.